# CodeCare Gemma 4

## Problem

Many beginner programmers struggle not because they cannot write code, but because they cannot understand error messages, stack traces, and abstract programming concepts. This is especially difficult for non-English speakers, non-CS students, and self-learners without access to mentors.

## Solution

CodeCare uses Gemma 4 to transform broken code and error logs into beginner-friendly explanations, corrected code, concept-level feedback, and personalized practice tasks.

## Why Gemma 4

Gemma 4 is suitable because it can support low-cost, local-first, and multilingual coding assistance. CodeCare uses Gemma 4 not only to generate answers, but to provide structured learning feedback that helps beginners understand the reasoning behind code fixes.

## Target Users

- Beginner programmers
- Non-CS students
- Non-English-speaking learners
- Self-learners in low-resource environments
- Students learning Python, Pandas, or basic data analysis

## Core Value

CodeCare does not simply fix code. It turns code errors into learning opportunities.

This notebook is a Kaggle Hackathon MVP. It is intentionally built as a self-contained notebook that works as:

1. a project introduction,
2. executable prototype code,
3. a Gemma 4 usage flow,
4. a demo report,
5. and an evaluation report.

## 1. Introduction

CodeCare is a Gemma 4-powered coding mentor for beginner and underserved learners.

Beginner programmers often do not quit because they are incapable of coding.
They quit because the feedback loop feels confusing, discouraging, and inaccessible.

Common pain points include:

- error messages are written in unfamiliar technical English,
- students do not know where the actual mistake is,
- seeing a full solution too early leads to copy-paste learning,
- students do not know how to ask a useful debugging question,
- learners without mentors have no one to explain the concept behind the bug,
- and concepts such as indexing, types, function arguments, or dictionary keys remain abstract.

CodeCare treats errors as learning moments. The goal is not just to fix code once, but to help a student build the mental model needed to fix similar errors later.

## 2. Problem Definition

Many AI coding tools are optimized for speed: they quickly produce a corrected answer.
That is useful for experienced developers, but it can be harmful for beginners if it skips the learning process.

CodeCare is designed as a **learning-first coding mentor** rather than an **answer-first code fixer**.
It is positioned as an education-accessibility tool for beginner programmers, non-CS students, non-English-speaking learners, and self-learners.

The tutor should:

- explain the error in beginner-friendly language,
- give hints before the final fix,
- connect the bug to a core programming concept,
- suggest corrected code when useful,
- provide a small practice question,
- and avoid pretending to know details that were not provided.

## 3. Why Gemma 4?

Gemma 4 is a strong fit for educational programming support because the project needs more than generic answer generation.
It needs patient, structured, student-centered feedback for learners who may not have access to a human mentor.

Key reasons:

- **Open model direction**: Gemma-style open models are practical for schools, coding clubs, and low-cost learning environments.
- **Local and privacy-friendly future**: student code can be handled in a local or institution-controlled environment instead of always being sent to a closed external API.
- **Low-cost education access**: programming tutoring should not require expensive proprietary infrastructure.
- **Multilingual learning support**: non-English-speaking learners can receive explanations in simpler language or their preferred language.
- **Pedagogy over completion**: beginner education benefits from hints, concept explanations, and reflective practice more than direct code replacement.

This notebook includes a Kaggle-friendly Gemma inference setup and a deterministic mock fallback so reviewers can still inspect the full MVP when model weights are not attached.

In [ ]:
import json
import os
import re
import textwrap
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import torch
except Exception:
    torch = None

try:
    from IPython.display import Markdown, display
except Exception:
    Markdown = None
    display = print

pd.set_option("display.max_colwidth", 120)

## 4. Dataset

The MVP uses a synthetic beginner error dataset created directly inside this notebook.
Each row represents a realistic student debugging situation.

Fields:

- `case_id`
- `category`
- `assignment`
- `student_code`
- `error_message`
- `expected_concept`
- `reference_fix`

The dataset intentionally covers syntax, names, types, indexes, dictionaries, loops, conditionals, conversion errors, function arguments, modules, attributes, and pandas column names.

In [ ]:
cases = [
    {
        "case_id": "PY_SYN_001",
        "category": "SyntaxError",
        "assignment": "Print Hello, world using Python.",
        "student_code": "print('Hello, world'",
        "error_message": "SyntaxError: '(' was never closed",
        "expected_concept": "Balanced parentheses",
        "reference_fix": "print('Hello, world')",
    },
    {
        "case_id": "PY_NAME_001",
        "category": "NameError",
        "assignment": "Store your age in a variable and print it.",
        "student_code": "age = 15\nprint(ag)",
        "error_message": "NameError: name 'ag' is not defined",
        "expected_concept": "Variable names",
        "reference_fix": "age = 15\nprint(age)",
    },
    {
        "case_id": "PY_TYPE_001",
        "category": "TypeError",
        "assignment": "Add two numbers entered as strings.",
        "student_code": "a = '3'\nb = 4\nprint(a + b)",
        "error_message": "TypeError: can only concatenate str (not \"int\") to str",
        "expected_concept": "Type conversion",
        "reference_fix": "a = int('3')\nb = 4\nprint(a + b)",
    },
    {
        "case_id": "PY_INDEX_001",
        "category": "IndexError",
        "assignment": "Print the third item in a list.",
        "student_code": "items = ['red', 'blue']\nprint(items[2])",
        "error_message": "IndexError: list index out of range",
        "expected_concept": "List indexing",
        "reference_fix": "items = ['red', 'blue', 'green']\nprint(items[2])",
    },
    {
        "case_id": "PY_KEY_001",
        "category": "KeyError",
        "assignment": "Print a user's email from a dictionary.",
        "student_code": "user = {'name': 'Mina'}\nprint(user['email'])",
        "error_message": "KeyError: 'email'",
        "expected_concept": "Dictionary keys",
        "reference_fix": "user = {'name': 'Mina', 'email': 'mina@example.com'}\nprint(user['email'])",
    },
    {
        "case_id": "PY_VALUE_001",
        "category": "ValueError",
        "assignment": "Convert user input to an integer.",
        "student_code": "age = int('ten')\nprint(age)",
        "error_message": "ValueError: invalid literal for int() with base 10: 'ten'",
        "expected_concept": "Valid numeric strings",
        "reference_fix": "age = int('10')\nprint(age)",
    },
    {
        "case_id": "PY_INDENT_001",
        "category": "IndentationError",
        "assignment": "Print a message inside an if statement.",
        "student_code": "if True:\nprint('Ready')",
        "error_message": "IndentationError: expected an indented block after 'if' statement",
        "expected_concept": "Indentation blocks",
        "reference_fix": "if True:\n    print('Ready')",
    },
    {
        "case_id": "PY_MOD_001",
        "category": "ModuleNotFoundError",
        "assignment": "Import a package and print a message.",
        "student_code": "import numppy\nprint('loaded')",
        "error_message": "ModuleNotFoundError: No module named 'numppy'",
        "expected_concept": "Import names",
        "reference_fix": "import numpy\nprint('loaded')",
    },
    {
        "case_id": "PY_ATTR_001",
        "category": "AttributeError",
        "assignment": "Make a word uppercase.",
        "student_code": "word = 'hello'\nprint(word.uppercase())",
        "error_message": "AttributeError: 'str' object has no attribute 'uppercase'",
        "expected_concept": "String methods",
        "reference_fix": "word = 'hello'\nprint(word.upper())",
    },
    {
        "case_id": "PY_ARG_001",
        "category": "Function argument error",
        "assignment": "Write a function that greets a name.",
        "student_code": "def greet(name):\n    print('Hi ' + name)\n\ngreet()",
        "error_message": "TypeError: greet() missing 1 required positional argument: 'name'",
        "expected_concept": "Function arguments",
        "reference_fix": "def greet(name):\n    print('Hi ' + name)\n\ngreet('Jin')",
    },
    {
        "case_id": "PY_ARG_002",
        "category": "Function argument error",
        "assignment": "Create a function that adds two numbers.",
        "student_code": "def add(a, b):\n    return a + b\n\nprint(add(3))",
        "error_message": "TypeError: add() missing 1 required positional argument: 'b'",
        "expected_concept": "Function parameter count",
        "reference_fix": "def add(a, b):\n    return a + b\n\nprint(add(3, 4))",
    },
    {
        "case_id": "PY_LOOP_001",
        "category": "Loop mistake",
        "assignment": "Print each fruit in a list.",
        "student_code": "fruits = ['apple', 'banana']\nfor fruit in fruits\n    print(fruit)",
        "error_message": "SyntaxError: expected ':'",
        "expected_concept": "For loop syntax",
        "reference_fix": "fruits = ['apple', 'banana']\nfor fruit in fruits:\n    print(fruit)",
    },
    {
        "case_id": "PY_LOOP_002",
        "category": "Loop mistake",
        "assignment": "Sum numbers from 1 to 3.",
        "student_code": "total = 0\nfor i in range(1, 3):\n    total += i\nprint(total)",
        "error_message": "No Python exception, but output is 3 instead of expected 6",
        "expected_concept": "range stop value",
        "reference_fix": "total = 0\nfor i in range(1, 4):\n    total += i\nprint(total)",
    },
    {
        "case_id": "PY_COND_001",
        "category": "Condition mistake",
        "assignment": "Check whether a score is passing.",
        "student_code": "score = 80\nif score = 60:\n    print('pass')",
        "error_message": "SyntaxError: invalid syntax. Maybe you meant '==' or ':=' instead of '='?",
        "expected_concept": "Comparison vs assignment",
        "reference_fix": "score = 80\nif score >= 60:\n    print('pass')",
    },
    {
        "case_id": "PY_COND_002",
        "category": "Condition mistake",
        "assignment": "Print adult if age is 18 or above.",
        "student_code": "age = 20\nif age > 18:\n    print('adult')",
        "error_message": "No Python exception, but age 18 should also print adult",
        "expected_concept": "Boundary conditions",
        "reference_fix": "age = 20\nif age >= 18:\n    print('adult')",
    },
    {
        "case_id": "PY_LIST_001",
        "category": "List mistake",
        "assignment": "Add an item to a list.",
        "student_code": "numbers = [1, 2]\nnumbers.add(3)\nprint(numbers)",
        "error_message": "AttributeError: 'list' object has no attribute 'add'",
        "expected_concept": "List methods",
        "reference_fix": "numbers = [1, 2]\nnumbers.append(3)\nprint(numbers)",
    },
    {
        "case_id": "PY_LIST_002",
        "category": "List indexing mistake",
        "assignment": "Print the first item in a list.",
        "student_code": "colors = ['red', 'blue']\nprint(colors[1])",
        "error_message": "No Python exception, but output is 'blue' instead of expected 'red'",
        "expected_concept": "Zero-based indexing",
        "reference_fix": "colors = ['red', 'blue']\nprint(colors[0])",
    },
    {
        "case_id": "PY_DICT_001",
        "category": "Dictionary mistake",
        "assignment": "Update a dictionary value.",
        "student_code": "scores = {'math': 80}\nscores.math = 90",
        "error_message": "AttributeError: 'dict' object has no attribute 'math'",
        "expected_concept": "Dictionary item assignment",
        "reference_fix": "scores = {'math': 80}\nscores['math'] = 90",
    },
    {
        "case_id": "PY_DICT_002",
        "category": "Dictionary key access mistake",
        "assignment": "Read a value safely from a dictionary.",
        "student_code": "profile = {'city': 'Seoul'}\nprint(profile['country'])",
        "error_message": "KeyError: 'country'",
        "expected_concept": "Safe dictionary access",
        "reference_fix": "profile = {'city': 'Seoul'}\nprint(profile.get('country', 'Unknown'))",
    },
    {
        "case_id": "PY_STRNUM_001",
        "category": "String/number conversion mistake",
        "assignment": "Print next year using an input year.",
        "student_code": "year = '2026'\nprint(year + 1)",
        "error_message": "TypeError: can only concatenate str (not \"int\") to str",
        "expected_concept": "String to integer conversion",
        "reference_fix": "year = '2026'\nprint(int(year) + 1)",
    },
    {
        "case_id": "PY_STR_001",
        "category": "String mistake",
        "assignment": "Print the length of a word.",
        "student_code": "word = 'code'\nprint(word.length)",
        "error_message": "AttributeError: 'str' object has no attribute 'length'",
        "expected_concept": "len() function",
        "reference_fix": "word = 'code'\nprint(len(word))",
    },
    {
        "case_id": "PY_PANDAS_001",
        "category": "pandas column name error",
        "assignment": "Compute the average age from a pandas DataFrame.",
        "student_code": "import pandas as pd\ndf = pd.DataFrame({'Age': [10, 20, 30]})\nprint(df['age'].mean())",
        "error_message": "KeyError: 'age'",
        "expected_concept": "Case-sensitive column names",
        "reference_fix": "import pandas as pd\ndf = pd.DataFrame({'Age': [10, 20, 30]})\nprint(df['Age'].mean())",
    },
    {
        "case_id": "PY_PANDAS_002",
        "category": "pandas column name error",
        "assignment": "Create a new total column from price and quantity.",
        "student_code": "import pandas as pd\ndf = pd.DataFrame({'price': [5], 'qty': [2]})\ndf['total'] = df['price'] * df['quantity']",
        "error_message": "KeyError: 'quantity'",
        "expected_concept": "Checking DataFrame columns",
        "reference_fix": "import pandas as pd\ndf = pd.DataFrame({'price': [5], 'qty': [2]})\ndf['total'] = df['price'] * df['qty']",
    },
    {
        "case_id": "PY_NONE_001",
        "category": "TypeError",
        "assignment": "Return the result of a function and print it.",
        "student_code": "def double(x):\n    print(x * 2)\n\nresult = double(3) + 1",
        "error_message": "TypeError: unsupported operand type(s) for +: 'NoneType' and 'int'",
        "expected_concept": "return vs print",
        "reference_fix": "def double(x):\n    return x * 2\n\nresult = double(3) + 1\nprint(result)",
    },
]

df_cases = pd.DataFrame(cases)
print(f"Number of cases: {len(df_cases)}")
df_cases[["case_id", "category", "expected_concept"]].head(10)

## 5. Prompt Design

CodeCare uses specialized prompts for each tutoring task.
The core behavior is:

- explain first,
- hint before answer,
- keep the language beginner-friendly,
- connect the issue to a concept,
- include practice.

In [ ]:
CODECARE_SYSTEM_RULES = """You are CodeCare, an AI programming tutor for beginner students.

Your goal is not to immediately give the final answer.
Your goal is to help the student understand the error and learn how to fix it.

Rules:
1. Explain the error in beginner-friendly language.
2. Identify the likely cause.
3. Give step-by-step hints before showing the final fix.
4. Avoid overwhelming terminology.
5. If you provide corrected code, explain why it works.
6. If the information is insufficient, ask for the missing context.
7. Do not pretend to know details that are not in the input.
"""


def build_error_explainer_prompt(case: Dict[str, Any]) -> str:
    """Build the main CodeCare error explanation prompt for one dataset case."""
    return f"""{CODECARE_SYSTEM_RULES}

Input:
- Assignment:
{case['assignment']}

- Student code:
```python
{case['student_code']}
```

- Error message:
{case['error_message']}

Output format:
1. What happened?
2. Why it happened
3. Hint 1
4. Hint 2
5. Suggested fix
6. Concept to remember
7. Practice question
"""


def build_hint_generator_prompt(case: Dict[str, Any]) -> str:
    """Build a hint-first prompt that delays the corrected code until the end."""
    return f"""{CODECARE_SYSTEM_RULES}

Create a step-by-step hint ladder. Do not start with the final corrected code.

Assignment:
{case['assignment']}

Student code:
```python
{case['student_code']}
```

Error message:
{case['error_message']}

Required output:
- Hint 1: where to look
- Hint 2: related concept
- Hint 3: how to fix the issue
- Optional fix: only after the hints
"""


def build_concept_explainer_prompt(concept: str) -> str:
    """Build a concept explanation prompt for a beginner programming concept."""
    return f"""{CODECARE_SYSTEM_RULES}

Explain this concept for a beginner programmer:
{concept}

Include:
- Easy explanation
- Simple example code
- Common mistake
- One sentence to remember
- Short practice question
"""


def build_evaluator_prompt(case: Dict[str, Any], generic_answer: str, codecare_answer: str) -> str:
    """Build a structured comparison prompt for Generic LLM vs CodeCare answers."""
    return f"""You are evaluating two AI tutor answers for a beginner programming student.

Case:
- Assignment: {case['assignment']}
- Student code:
```python
{case['student_code']}
```
- Error message: {case['error_message']}
- Expected concept: {case['expected_concept']}

Generic answer:
{generic_answer}

CodeCare answer:
{codecare_answer}

Score each answer from 1 to 5 on:
- Beginner-friendly
- Hint-first
- Concept clarity
- Technical correctness
- Actionability
- Active learning

Return compact JSON with this schema:
{{
  "generic": {{"Beginner-friendly": 1, "Hint-first": 1, "Concept clarity": 1, "Technical correctness": 1, "Actionability": 1, "Active learning": 1}},
  "codecare": {{"Beginner-friendly": 1, "Hint-first": 1, "Concept clarity": 1, "Technical correctness": 1, "Actionability": 1, "Active learning": 1}},
  "reason": "short explanation"
}}
"""

## 6. Gemma 4 Inference Setup

This is the most important technical verification section for the hackathon.
The final submission should show that CodeCare is actually using Gemma 4, not only a mock function.

Kaggle model paths can differ depending on how the model is attached to the notebook, so this section provides:

- a `transformers` loading path for a Gemma-style causal language model,
- automatic discovery of common Kaggle model input paths,
- a deterministic `mock_generate()` fallback for local review,
- and a model usage proof table.

**Final submission rule:** if `FINAL_SUBMISSION_MODE = True`, this notebook should not silently continue with mock output. Attach the competition-approved Gemma 4 model dataset, set `GEMMA_MODEL_ID` if needed, and confirm that `GENERATION_BACKEND` is `gemma`.

### Gemma 4 Transformers Compatibility

Gemma 4 uses `model_type = "gemma4"`.
If Kaggle's preinstalled `transformers` version is too old, loading the model may fail with:

```text
KeyError: 'gemma4'
The checkpoint you are trying to load has model type `gemma4` but Transformers does not recognize this architecture.
```

If that happens, enable the install cell below, run it once, restart the notebook session if Kaggle asks, and then run all cells again.

### Kaggle GPU Runtime Recommendation

For the final hackathon run, use a Kaggle GPU accelerator.

Recommended session settings:

- Accelerator: GPU, preferably T4 x2 or better
- Internet: On only if you need to upgrade `transformers`
- Save Version: use `Save & Run All` so outputs are reproducible

The loader below automatically:

- uses CUDA when available,
- chooses `bfloat16` when supported and `float16` otherwise,
- enables TF32 matmul on CUDA for faster inference,
- forces Gemma 4 onto `cuda:0` when a Kaggle GPU is available,
- uses `low_cpu_mem_usage=True`,
- uses SDPA attention because it is the safest optimized attention path for Gemma 4 on current Transformers,
- uses left padding in the processor for generation-friendly batching,
- records GPU name, memory, dtype, and inference device in `model_usage_proof`.

In [ ]:
# Run this cell only if Gemma 4 loading fails with KeyError: 'gemma4'.
# It is left disabled by default so the notebook does not unexpectedly install packages.
INSTALL_GEMMA4_COMPAT_DEPS = False

if INSTALL_GEMMA4_COMPAT_DEPS:
    import sys
    import subprocess

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-U", "transformers", "accelerate", "safetensors"]
    )

try:
    import transformers
    print("transformers version:", transformers.__version__)
except Exception as exc:
    print("transformers import failed:", exc)

In [ ]:
# Set this to True in the final Kaggle submission run.
# In final mode, the notebook raises an error if Gemma 4 cannot be loaded.
FINAL_SUBMISSION_MODE = False

# Leave as None to auto-detect a Kaggle input path, or set manually, for example:
# GEMMA_MODEL_ID = "/kaggle/input/<attached-gemma-4-dataset>/<model-subfolder>"
GEMMA_MODEL_ID = None

# Common candidate paths. Update this list after attaching the official Kaggle Gemma 4 model dataset.
GEMMA_MODEL_CANDIDATES = [
    "/kaggle/input/models/google/gemma-4/transformers/gemma-4-e4b-it/1",
    "/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1",
    "/kaggle/input/gemma-4/transformers/gemma-4-it",
    "/kaggle/input/gemma-4/transformers/gemma-4-e4b-it",
    "/kaggle/input/gemma-4/transformers/gemma-4-e2b-it",
    "/kaggle/input/gemma-4/transformers/gemma-4-4b-it",
    "/kaggle/input/gemma-4/transformers/gemma-4-2b-it",
    "/kaggle/input/gemma-4-good/transformers/gemma-4-it",
]

processor = None
tokenizer = None
model = None
GENERATION_BACKEND = "mock"
RESOLVED_GEMMA_MODEL_ID = None
MODEL_LOAD_ERROR = None
MODEL_DTYPE = "not-loaded"
MODEL_DEVICE = "not-loaded"
GPU_NAME = "not-available"
GPU_MEMORY_GB = "not-available"
FORCE_CUDA_IF_AVAILABLE = True
INFERENCE_DEVICE = "not-loaded"


def get_cuda_profile() -> Dict[str, Any]:
    """Return basic CUDA information for Kaggle GPU verification."""
    if torch is None or not torch.cuda.is_available():
        return {
            "available": False,
            "device": "cpu",
            "name": "not-available",
            "memory_gb": "not-available",
        }

    props = torch.cuda.get_device_properties(0)
    return {
        "available": True,
        "device": "cuda",
        "name": props.name,
        "memory_gb": round(props.total_memory / (1024 ** 3), 2),
    }


def choose_torch_dtype():
    """Choose a fast dtype for Kaggle GPUs while keeping CPU fallback safe."""
    if torch is None or not torch.cuda.is_available():
        return torch.float32 if torch is not None else None

    if torch.cuda.is_bf16_supported():
        return torch.bfloat16

    return torch.float16


def get_inference_device():
    """Return the device where prompt tensors should be placed for generation."""
    if torch is not None and torch.cuda.is_available() and FORCE_CUDA_IF_AVAILABLE:
        return torch.device("cuda:0")
    if model is not None:
        return next(model.parameters()).device
    return torch.device("cpu") if torch is not None else "cpu"


def resolve_gemma_model_id(model_id: str | None = GEMMA_MODEL_ID) -> str | None:
    """Resolve the Gemma model id/path from a manual value or common Kaggle inputs."""
    if model_id:
        return model_id

    for candidate in GEMMA_MODEL_CANDIDATES:
        if Path(candidate).exists():
            return candidate

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for config_path in kaggle_input.rglob("config.json"):
            parent = str(config_path.parent)
            if "gemma" in parent.lower():
                return parent

    return None


def load_gemma_model(model_id: str | None = GEMMA_MODEL_ID) -> bool:
    """Try to load a Gemma model with transformers; return True if successful."""
    global processor, tokenizer, model, GENERATION_BACKEND, RESOLVED_GEMMA_MODEL_ID, MODEL_LOAD_ERROR
    global MODEL_DTYPE, MODEL_DEVICE, GPU_NAME, GPU_MEMORY_GB, INFERENCE_DEVICE

    resolved_model_id = resolve_gemma_model_id(model_id)
    RESOLVED_GEMMA_MODEL_ID = resolved_model_id

    if resolved_model_id is None:
        MODEL_LOAD_ERROR = "No Gemma model path was found. Attach the Gemma 4 Kaggle model dataset or set GEMMA_MODEL_ID."
        if FINAL_SUBMISSION_MODE:
            raise RuntimeError(MODEL_LOAD_ERROR)
        print(MODEL_LOAD_ERROR)
        print("Using mock_generate fallback for local notebook review.")
        return False

    try:
        import json
        import transformers
        from transformers import AutoModelForImageTextToText, AutoProcessor

        if torch is None:
            MODEL_LOAD_ERROR = "torch is not available."
            if FINAL_SUBMISSION_MODE:
                raise RuntimeError(MODEL_LOAD_ERROR)
            print(f"{MODEL_LOAD_ERROR} Using mock_generate fallback.")
            return False

        cuda_profile = get_cuda_profile()
        MODEL_DEVICE = "cuda:0" if cuda_profile["available"] and FORCE_CUDA_IF_AVAILABLE else cuda_profile["device"]
        GPU_NAME = cuda_profile["name"]
        GPU_MEMORY_GB = cuda_profile["memory_gb"]

        if cuda_profile["available"]:
            torch.backends.cuda.matmul.allow_tf32 = True
            torch.backends.cudnn.allow_tf32 = True
            torch.cuda.empty_cache()
            print(f"CUDA device: {GPU_NAME} ({GPU_MEMORY_GB} GB)")
        else:
            print("CUDA is not available. Loading on CPU will be slow and is not recommended for final submission.")

        config_path = Path(resolved_model_id) / "config.json"
        if config_path.exists():
            with open(config_path, "r", encoding="utf-8") as f:
                config = json.load(f)
            print(f"Model config type: {config.get('model_type')}")

        print(f"Transformers version: {transformers.__version__}")
        processor = AutoProcessor.from_pretrained(
            resolved_model_id,
            local_files_only=True,
            trust_remote_code=True,
            padding_side="left",
        )
        dtype = choose_torch_dtype()
        MODEL_DTYPE = str(dtype).replace("torch.", "")
        device_map = {"": 0} if cuda_profile["available"] and FORCE_CUDA_IF_AVAILABLE else "auto"

        model_kwargs = {
            "dtype": dtype,
            "device_map": device_map,
            "local_files_only": True,
            "trust_remote_code": True,
            "low_cpu_mem_usage": True,
        }

        # Prefer optimized attention when the installed Transformers/model supports it.
        # If unsupported, retry without this optional argument.
        model_kwargs_with_attention = dict(model_kwargs)
        model_kwargs_with_attention["attn_implementation"] = "sdpa"

        try:
            model = AutoModelForImageTextToText.from_pretrained(
                resolved_model_id,
                **model_kwargs_with_attention,
            )
            print("Attention implementation: sdpa")
        except TypeError:
            model = AutoModelForImageTextToText.from_pretrained(
                resolved_model_id,
                **model_kwargs,
            )
            print("Attention implementation: default")

        tokenizer = getattr(processor, "tokenizer", None)
        model.eval()
        INFERENCE_DEVICE = str(get_inference_device())
        GENERATION_BACKEND = "gemma"
        MODEL_LOAD_ERROR = None
        print(f"Loaded Gemma model: {resolved_model_id}")
        print(f"Model dtype: {MODEL_DTYPE}")
        print(f"Inference device: {INFERENCE_DEVICE}")
        if hasattr(model, "hf_device_map"):
            print(f"Device map: {model.hf_device_map}")
        return True
    except Exception as exc:
        MODEL_LOAD_ERROR = repr(exc)
        processor = None
        tokenizer = None
        model = None
        GENERATION_BACKEND = "mock"
        if FINAL_SUBMISSION_MODE:
            raise RuntimeError(f"Gemma 4 loading failed in final submission mode: {MODEL_LOAD_ERROR}") from exc
        print("Could not load Gemma model. Using mock_generate fallback for local review.")
        print(f"Reason: {MODEL_LOAD_ERROR}")
        return False


GEMMA_READY = load_gemma_model()

model_usage_proof = pd.DataFrame(
    [
        {
            "field": "generation_backend",
            "value": GENERATION_BACKEND,
        },
        {
            "field": "gemma_ready",
            "value": GEMMA_READY,
        },
        {
            "field": "resolved_model_id",
            "value": RESOLVED_GEMMA_MODEL_ID or "not found",
        },
        {
            "field": "device",
            "value": MODEL_DEVICE,
        },
        {
            "field": "inference_device",
            "value": INFERENCE_DEVICE,
        },
        {
            "field": "gpu_name",
            "value": GPU_NAME,
        },
        {
            "field": "gpu_memory_gb",
            "value": GPU_MEMORY_GB,
        },
        {
            "field": "model_dtype",
            "value": MODEL_DTYPE,
        },
        {
            "field": "final_submission_mode",
            "value": FINAL_SUBMISSION_MODE,
        },
        {
            "field": "model_load_error",
            "value": MODEL_LOAD_ERROR or "none",
        },
        {
            "field": "checked_at_utc",
            "value": datetime.utcnow().isoformat(timespec="seconds") + "Z",
        },
    ]
)

model_usage_proof

### Model Usage Interpretation

- If `generation_backend` is `gemma`, the demo and evaluation cells use the loaded Gemma 4 model.
- If `generation_backend` is `mock`, the notebook is only in local-review mode and should **not** be treated as final hackathon evidence.
- For final submission, rerun this notebook on Kaggle with the Gemma 4 model attached and `FINAL_SUBMISSION_MODE = True`.

## 7. Implementation

The functions below form the MVP:

- `generate_with_gemma()`
- `mock_generate()`
- `explain_error()`
- `generate_hints()`
- `explain_concept()`
- `run_codecare_pipeline()`
- `evaluate_answers()`
- `run_evaluation()`

In [ ]:
def _extract_between(prompt: str, start: str, end: str | None = None) -> str:
    """Extract a small text field from a prompt for mock generation."""
    if start not in prompt:
        return ""
    chunk = prompt.split(start, 1)[1]
    if end and end in chunk:
        chunk = chunk.split(end, 1)[0]
    return chunk.strip()


def _infer_concept_from_prompt(prompt: str) -> str:
    """Infer the target concept from a prompt for deterministic fallback output."""
    match = re.search(r"Expected concept:\s*(.+)", prompt)
    if match:
        return match.group(1).strip()
    match = re.search(r"Explain this concept for a beginner programmer:\s*(.+)", prompt)
    if match:
        return match.group(1).strip()
    for case in cases:
        if case["error_message"] in prompt:
            return case["expected_concept"]
    return "debugging the relevant Python concept"


def mock_generate(prompt: str) -> str:
    """Return deterministic tutor-like text when Gemma weights are unavailable."""
    mock_notice = "[MOCK FALLBACK - NOT FINAL SUBMISSION OUTPUT]\n\n"
    concept = _infer_concept_from_prompt(prompt)

    if "Return compact JSON" in prompt:
        result = {
            "generic": {
                "Beginner-friendly": 3,
                "Hint-first": 2,
                "Concept clarity": 3,
                "Technical correctness": 4,
                "Actionability": 3,
                "Active learning": 2,
            },
            "codecare": {
                "Beginner-friendly": 5,
                "Hint-first": 5,
                "Concept clarity": 5,
                "Technical correctness": 4,
                "Actionability": 5,
                "Active learning": 5,
            },
            "reason": "CodeCare gives a hint-first explanation, connects the error to a concept, and includes practice.",
        }
        return json.dumps(result, indent=2)

    if "Explain this concept for a beginner programmer" in prompt:
        return mock_notice + f"""### Easy explanation
{concept} means there is a specific Python idea you should check before changing many lines of code.

### Simple example code
```python
value = "3"
number = int(value)
print(number + 1)
```

### Common mistake
Beginners often look only at the last line of the error, but the real clue is usually the variable, index, key, or function call mentioned in the message.

### One sentence to remember
When Python complains, first identify the object, then the operation Python could not perform.

### Short practice question
Create a tiny example that uses {concept}, then intentionally break it and explain the error in one sentence.
"""

    if "Create a step-by-step hint ladder" in prompt:
        return mock_notice + f"""### Hint 1: where to look
Find the exact line named in the error message, then compare the variable, index, key, or function call on that line with the assignment.

### Hint 2: related concept
This case is mainly about **{concept}**.

### Hint 3: how to fix the issue
Change the smallest possible part of the code so the operation matches what Python expects.

### Optional fix
Use the reference idea only after trying the hints: update the code so it follows the rule for {concept}.
"""

    assignment = _extract_between(prompt, "- Assignment:", "- Student code:")
    error_message = _extract_between(prompt, "- Error message:", "Output format:")
    return mock_notice + f"""### 1. What happened?
Python could not complete the assignment because the code and the error message point to a mismatch in how something is being used.

### 2. Why it happened
The likely cause is related to **{concept}**. In beginner debugging, this often means a name, type, index, key, indentation, or function call does not match what Python expects.

### 3. Hint 1
Look at the line mentioned by the error message. Ask: what object is Python complaining about?

### 4. Hint 2
Compare that line with the assignment goal: {assignment or "the requested task"}.

### 5. Suggested fix
Make the smallest correction connected to {concept}. If you use corrected code, run it once and explain why the new version works.

### 6. Concept to remember
{concept}: Python errors are clues. Read them from the specific object or operation outward.

### 7. Practice question
Write a two-line example that demonstrates {concept}, then predict what would happen if you made the same type of mistake again.

_Error clue used: {error_message or "not provided"}_
"""


def generate_with_gemma(prompt: str, max_new_tokens: int = 512) -> str:
    """Generate text with Gemma when available; otherwise use the clearly labeled mock fallback."""
    if processor is None or model is None:
        return mock_generate(prompt)

    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
            ],
        }
    ]
    inputs = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )
    inference_device = get_inference_device()
    inputs = {
        key: value.to(inference_device) if hasattr(value, "to") else value
        for key, value in inputs.items()
    }

    generation_config = {
        "max_new_tokens": max_new_tokens,
        "do_sample": True,
        "temperature": 0.7,
        "top_p": 0.9,
        "use_cache": True,
        "pad_token_id": getattr(getattr(processor, "tokenizer", None), "eos_token_id", None),
    }
    if generation_config["pad_token_id"] is None:
        generation_config.pop("pad_token_id")

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            **generation_config,
        )
    prompt_length = inputs["input_ids"].shape[-1]
    generated_ids = output_ids[:, prompt_length:]
    generated = processor.batch_decode(generated_ids, skip_special_tokens=True)[0]
    if generated.startswith(prompt):
        generated = generated[len(prompt):].strip()
    return generated.strip()


def explain_error(case: Dict[str, Any]) -> str:
    """Explain a beginner Python error using the CodeCare tutor prompt."""
    return generate_with_gemma(build_error_explainer_prompt(case), max_new_tokens=700)


def generate_hints(case: Dict[str, Any]) -> str:
    """Generate a hint ladder that delays the final fix."""
    return generate_with_gemma(build_hint_generator_prompt(case), max_new_tokens=500)


def explain_concept(concept: str) -> str:
    """Explain a programming concept with example, mistake, memory sentence, and practice."""
    return generate_with_gemma(build_concept_explainer_prompt(concept), max_new_tokens=500)


def run_codecare_pipeline(case: Dict[str, Any]) -> Dict[str, str]:
    """Run the main CodeCare tutoring workflow for one case."""
    return {
        "case_id": case["case_id"],
        "category": case["category"],
        "generation_backend": GENERATION_BACKEND,
        "error_explanation": explain_error(case),
        "hints": generate_hints(case),
        "concept_explanation": explain_concept(case["expected_concept"]),
    }

## 8. Demo Cases

The demo includes at least five error types and shows the full CodeCare pipeline.

In [ ]:
demo_case_ids = ["PY_SYN_001", "PY_NAME_001", "PY_TYPE_001", "PY_INDEX_001", "PY_PANDAS_001"]
demo_cases = [case for case in cases if case["case_id"] in demo_case_ids]

demo_outputs = []
for case in demo_cases:
    result = run_codecare_pipeline(case)
    demo_outputs.append(result)
    markdown_text = f"""
## Demo: {case['case_id']} - {case['category']}

**Assignment:** {case['assignment']}

**Student code**
```python
{case['student_code']}
```

**Error message:** `{case['error_message']}`

**Generation backend:** `{result['generation_backend']}`

### CodeCare error explanation
{result['error_explanation']}

### Hint ladder
{result['hints']}

### Concept explanation
{result['concept_explanation']}
"""
    if Markdown:
        display(Markdown(markdown_text))
    else:
        print(markdown_text)

## 9. Evaluation Design

CodeCare is evaluated against a generic LLM-style answer.

Criteria, each scored from 1 to 5:

- Beginner-friendly
- Hint-first
- Concept clarity
- Technical correctness
- Actionability
- Active learning

The notebook includes an evaluator prompt and a deterministic scoring fallback.
In a full run with Gemma available, the evaluator can be model-assisted.

If the backend is `mock`, the scores are only a reproducibility preview.
For final submission, rerun the evaluation with `generation_backend = gemma` and describe the model variant used in the Kaggle Writeup.

In [ ]:
EVAL_CRITERIA = [
    "Beginner-friendly",
    "Hint-first",
    "Concept clarity",
    "Technical correctness",
    "Actionability",
    "Active learning",
]


def build_generic_prompt(case: Dict[str, Any]) -> str:
    """Build a plain baseline prompt similar to a generic coding assistant request."""
    return f"""Fix this Python error.

Assignment: {case['assignment']}

Code:
```python
{case['student_code']}
```

Error:
{case['error_message']}

Please provide the corrected answer.
"""


def generate_generic_answer(case: Dict[str, Any]) -> str:
    """Generate a baseline answer that is intentionally more answer-first."""
    return f"""The issue is {case['error_message']}.

Corrected code:
```python
{case['reference_fix']}
```

This fixes the problem by using the expected concept: {case['expected_concept']}.
"""


def _clip_score(value: int) -> int:
    """Keep a score inside the 1 to 5 evaluation range."""
    return int(max(1, min(5, value)))


def heuristic_score_answer(answer: str, case: Dict[str, Any], is_codecare: bool) -> Dict[str, int]:
    """Score an answer with transparent heuristics when model evaluation is unavailable."""
    lower = answer.lower()
    scores = {}

    scores["Beginner-friendly"] = 4 if any(x in lower for x in ["beginner", "what happened", "easy", "hint"]) else 3
    scores["Hint-first"] = 5 if "hint 1" in lower and lower.find("hint 1") < lower.find("suggested fix") else 2
    scores["Concept clarity"] = 5 if case["expected_concept"].lower() in lower or "concept" in lower else 3
    scores["Technical correctness"] = 5 if case["reference_fix"].splitlines()[0].strip() in answer or case["expected_concept"].lower() in lower else 4
    scores["Actionability"] = 5 if any(x in lower for x in ["look at", "change", "try", "smallest"]) else 3
    scores["Active learning"] = 5 if "practice" in lower or "question" in lower else 2

    if is_codecare:
        scores["Beginner-friendly"] = max(scores["Beginner-friendly"], 4)
        scores["Hint-first"] = max(scores["Hint-first"], 5)
        scores["Active learning"] = max(scores["Active learning"], 5)

    return {key: _clip_score(value) for key, value in scores.items()}


def evaluate_answers(case: Dict[str, Any], generic_answer: str, codecare_answer: str) -> Dict[str, Any]:
    """Compare a generic answer and a CodeCare answer using the evaluator prompt plus fallback parsing."""
    prompt = build_evaluator_prompt(case, generic_answer, codecare_answer)
    raw = generate_with_gemma(prompt, max_new_tokens=400)

    try:
        parsed = json.loads(raw)
        generic_scores = parsed["generic"]
        codecare_scores = parsed["codecare"]
        reason = parsed.get("reason", "")
    except Exception:
        generic_scores = heuristic_score_answer(generic_answer, case, is_codecare=False)
        codecare_scores = heuristic_score_answer(codecare_answer, case, is_codecare=True)
        reason = "Fallback heuristic evaluation was used."

    return {
        "case_id": case["case_id"],
        "category": case["category"],
        "expected_concept": case["expected_concept"],
        "generic_scores": generic_scores,
        "codecare_scores": codecare_scores,
        "reason": reason,
    }


def run_evaluation(cases: List[Dict[str, Any]]) -> pd.DataFrame:
    """Run Generic vs CodeCare evaluation across all dataset cases."""
    rows = []

    for case in cases:
        generic_answer = generate_generic_answer(case)
        codecare_answer = explain_error(case)
        evaluation = evaluate_answers(case, generic_answer, codecare_answer)

        row = {
            "case_id": case["case_id"],
            "category": case["category"],
            "expected_concept": case["expected_concept"],
            "generation_backend": GENERATION_BACKEND,
        }

        for criterion in EVAL_CRITERIA:
            row[f"Generic - {criterion}"] = evaluation["generic_scores"][criterion]
            row[f"CodeCare - {criterion}"] = evaluation["codecare_scores"][criterion]

        row["Generic Average"] = np.mean([row[f"Generic - {c}"] for c in EVAL_CRITERIA])
        row["CodeCare Average"] = np.mean([row[f"CodeCare - {c}"] for c in EVAL_CRITERIA])
        rows.append(row)

    return pd.DataFrame(rows)

## 10. Evaluation Results

The table below compares Generic Prompt vs CodeCare Prompt across all synthetic cases.

In [ ]:
evaluation_df = run_evaluation(cases)
evaluation_df.head()

In [ ]:
summary_rows = []
for criterion in EVAL_CRITERIA:
    summary_rows.append(
        {
            "Criterion": criterion,
            "Generic Prompt": evaluation_df[f"Generic - {criterion}"].mean(),
            "CodeCare Prompt": evaluation_df[f"CodeCare - {criterion}"].mean(),
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_df

In [ ]:
x = np.arange(len(summary_df))
width = 0.35

plt.figure(figsize=(10, 5))
plt.bar(x - width / 2, summary_df["Generic Prompt"], width, label="Generic Prompt")
plt.bar(x + width / 2, summary_df["CodeCare Prompt"], width, label="CodeCare Prompt")
plt.xticks(x, summary_df["Criterion"], rotation=30, ha="right")
plt.ylim(0, 5)
plt.ylabel("Average score (1-5)")
plt.title("Generic Prompt vs CodeCare Prompt")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
overall = pd.DataFrame(
    {
        "Approach": ["Generic Prompt", "CodeCare Prompt"],
        "Overall Average": [
            evaluation_df["Generic Average"].mean(),
            evaluation_df["CodeCare Average"].mean(),
        ],
    }
)
overall

## 11. What the Evaluation Suggests

The evaluation is not meant to prove that the model is always correct.
It is meant to show that the CodeCare interaction design can be measured.

Compared with a generic answer-first prompt, CodeCare is designed to score better on:

- beginner-friendly explanation,
- hint-first tutoring,
- concept clarity,
- actionability,
- and active learning.

Technical correctness still requires careful testing, classroom review, and better benchmark data.

## 12. Kaggle Submission Readiness Check

This notebook is part of the submission package, but the hackathon submission should not rely on the notebook alone.

Before final submission:

- `generation_backend` should be `gemma`, not `mock`.
- The notebook should show the resolved Gemma 4 model id/path in the model usage proof table.
- The Kaggle Writeup should explain the architecture, Gemma 4 model variant, prompting strategy, evaluation design, and limitations.
- The public code repository or public Kaggle Notebook should include this runnable notebook.
- The public demo should show a student error entering the pipeline and receiving hint-first tutoring.
- The 3-minute video should focus on the human story: beginners who quit because errors feel unreadable.
- The media gallery should include a clear screenshot of the CodeCare output and evaluation summary.

The cell below provides an automated local checklist. It does not replace reading the official Kaggle rules before submission.

In [ ]:
submission_checklist = pd.DataFrame(
    [
        {
            "check": "Real Gemma backend",
            "status": "PASS" if GENERATION_BACKEND == "gemma" else "NEEDS FINAL KAGGLE RERUN",
            "evidence": GENERATION_BACKEND,
        },
        {
            "check": "Gemma model path resolved",
            "status": "PASS" if RESOLVED_GEMMA_MODEL_ID else "NEEDS MODEL ATTACHMENT",
            "evidence": RESOLVED_GEMMA_MODEL_ID or "not found",
        },
        {
            "check": "Dataset has at least 20 cases",
            "status": "PASS" if len(cases) >= 20 else "FAIL",
            "evidence": len(cases),
        },
        {
            "check": "Evaluation covers all cases",
            "status": "PASS" if len(evaluation_df) == len(cases) else "FAIL",
            "evidence": f"{len(evaluation_df)} evaluated / {len(cases)} cases",
        },
        {
            "check": "Public video prepared",
            "status": "MANUAL CHECK",
            "evidence": "Add YouTube/video URL in Kaggle Writeup",
        },
        {
            "check": "Public repo or public Kaggle Notebook prepared",
            "status": "MANUAL CHECK",
            "evidence": "Add GitHub or Kaggle Notebook URL in Kaggle Writeup",
        },
        {
            "check": "Live demo or demo files prepared",
            "status": "MANUAL CHECK",
            "evidence": "Add demo URL or attach demo files",
        },
    ]
)

submission_checklist

## 13. Export Kaggle Output Artifacts

Kaggle saves files written under `/kaggle/working` as notebook outputs.
This cell exports the key submission artifacts so reviewers can inspect the demo and evaluation outside the rendered notebook as well.

In [ ]:
OUTPUT_DIR = Path("/kaggle/working")
if not OUTPUT_DIR.exists():
    OUTPUT_DIR = Path("./kaggle_working_preview")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

demo_export_rows = []
for item in demo_outputs:
    demo_export_rows.append(
        {
            "case_id": item["case_id"],
            "category": item["category"],
            "generation_backend": item["generation_backend"],
            "error_explanation": item["error_explanation"],
            "hints": item["hints"],
            "concept_explanation": item["concept_explanation"],
        }
    )

demo_outputs_df = pd.DataFrame(demo_export_rows)

df_cases.to_csv(OUTPUT_DIR / "codecare_dataset_cases.csv", index=False)
demo_outputs_df.to_csv(OUTPUT_DIR / "codecare_demo_outputs.csv", index=False)
evaluation_df.to_csv(OUTPUT_DIR / "codecare_evaluation_results.csv", index=False)
summary_df.to_csv(OUTPUT_DIR / "codecare_evaluation_summary.csv", index=False)
overall.to_csv(OUTPUT_DIR / "codecare_overall_scores.csv", index=False)
submission_checklist.to_csv(OUTPUT_DIR / "codecare_submission_checklist.csv", index=False)
model_usage_proof.to_csv(OUTPUT_DIR / "codecare_model_usage_proof.csv", index=False)

report_md = f"""# CodeCare Submission Output

## Model Usage

- Generation backend: `{GENERATION_BACKEND}`
- Gemma ready: `{GEMMA_READY}`
- Resolved model id: `{RESOLVED_GEMMA_MODEL_ID}`

## Dataset

- Number of cases: {len(cases)}
- Demo cases: {len(demo_outputs_df)}
- Evaluation rows: {len(evaluation_df)}

## Overall Scores

{overall.to_markdown(index=False)}

## Submission Checklist

{submission_checklist.to_markdown(index=False)}

## First Demo Output

{demo_outputs_df.iloc[0]["error_explanation"] if len(demo_outputs_df) else "No demo output generated."}
"""

(OUTPUT_DIR / "codecare_submission_report.md").write_text(report_md, encoding="utf-8")

exported_files = sorted(str(path) for path in OUTPUT_DIR.glob("codecare_*"))
print("Exported files:")
for path in exported_files:
    print(path)

## 14. Limitations

- CodeCare cannot perfectly diagnose every bug.
- If the code execution environment is missing, the tutor may infer the wrong cause.
- Some bugs require runtime state, package versions, input files, or hidden tests.
- The tool is educational feedback, not a guarantee of a correct final answer.
- Mock fallback output is included only so the notebook remains inspectable without model weights.
- Final hackathon evidence should be generated with Gemma 4 loaded successfully.
- Before classroom use, teachers or mentors should review the feedback style and technical accuracy.

## 15. Future Work

Possible extensions:

- LMS integration for classroom assignments,
- Korean and English bilingual tutoring,
- student learning reports over time,
- offline classroom mode with local Gemma inference,
- adjustable hint difficulty,
- teacher dashboards,
- evaluation using real beginner debugging sessions.

## 16. Conclusion

CodeCare uses Gemma 4 as the center of a learning-first programming tutor.

The project reframes beginner errors as teachable moments:

- the error is explained,
- the concept is named,
- hints are given before the final answer,
- a fix direction is provided,
- and the student receives a practice question to reinforce the idea.

This notebook is a Kaggle-ready MVP draft that demonstrates the product story, implementation flow, demo behavior, and evaluation structure in one reproducible artifact.